<a href="https://colab.research.google.com/github/Abish-Devcraft/DSGP-GRP11-SPATIUM/blob/furniture-classifier/notebooks/06_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 Training and Evaluation
Input: furniture_v2_split <br>
Model: ResNet50 (ImageNet) fine-tuned for 17 classes, with augmentation and class weighting <br>
Output: trained model on Drive, validation and test results

In [8]:
# connect Drive, restore the split dataset, check the GPU
from google.colab import drive
drive.mount('/content/drive')

import torch, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

DRIVE = "/content/drive/MyDrive/DSGP"
!rm -rf /content/furniture_v2_split
!unzip -q -o {DRIVE}/furniture_v2_split.zip -d /content
ROOT = "/content/furniture_v2_split"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cuda


In [9]:
# train images get random changes; val/test are only resized and normalised
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

In [10]:
# load train/val/test; val and test reuse the train class numbers (test is missing 3 classes)
train_ds = datasets.ImageFolder(f"{ROOT}/train", train_tf)
val_ds   = datasets.ImageFolder(f"{ROOT}/val", eval_tf)
test_ds  = datasets.ImageFolder(f"{ROOT}/test", eval_tf)
CLASSES = train_ds.classes
for ds in (val_ds, test_ds):
    m = {i: train_ds.class_to_idx[c] for i, c in enumerate(ds.classes)}
    ds.samples = [(p, m[t]) for p, t in ds.samples]
    ds.targets = [t for _, t in ds.samples]

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=64, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=64, num_workers=2)
print(len(CLASSES), "classes |", len(train_ds), "train |", len(val_ds), "val |", len(test_ds), "test")

assert len(train_ds) == 2163 and len(val_ds) == 537 and len(test_ds) == 801, "folder counts don't match the manifest"

17 classes | 2163 train | 537 val | 801 test


In [11]:
# class weighting from Notebook 5: rarer classes count more in the loss
USE_WEIGHTS = True
counts = np.bincount(train_ds.targets, minlength=len(CLASSES))
weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float).to(device)
for c, n, w in zip(CLASSES, counts, weights): print(f"{c:16} {n:4}  weight {w:.2f}")

armchair          179  weight 0.71
bed               156  weight 0.82
bookshelf         111  weight 1.15
chair             161  weight 0.79
coffee_table      111  weight 1.15
console_table     111  weight 1.15
desk              109  weight 1.17
dresser           102  weight 1.25
dressing_table    119  weight 1.07
nightstand        107  weight 1.19
side_table         68  weight 1.87
sofa              202  weight 0.63
stool_ottoman     122  weight 1.04
storage_cabinet   132  weight 0.96
table              89  weight 1.43
tv_stand          106  weight 1.20
wardrobe          178  weight 0.71


In [12]:
# pretrained ResNet50: freeze most layers, retrain the last block, new 17-class output layer
EPOCHS = 12
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for p in model.parameters(): p.requires_grad = False
for p in model.layer4.parameters(): p.requires_grad = True
model.fc = torch.nn.Linear(2048, len(CLASSES))
model = model.to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=weights if USE_WEIGHTS else None)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
MODEL_PATH = f"{DRIVE}/furniture_resnet50_{'weighted' if USE_WEIGHTS else 'unweighted'}.pt"

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


In [13]:
# train for EPOCHS rounds; keep the version with the best validation macro F1
def evaluate(dl):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, t in dl:
            ps.append(model(x.to(device)).argmax(1).cpu()); ys.append(t)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

best, hist = 0, []
for ep in range(EPOCHS):
    model.train(); total = 0
    for x, t in train_dl:
        x, t = x.to(device), t.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), t)
        loss.backward(); opt.step()
        total += loss.item() * len(t)
    sched.step()
    yv, pv = evaluate(val_dl)
    f1 = f1_score(yv, pv, average="macro")
    hist.append((total / len(train_ds), f1))
    if f1 > best:
        best = f1; torch.save(model.state_dict(), MODEL_PATH)
    print(f"epoch {ep + 1:2}  train loss {hist[-1][0]:.3f}  val macro F1 {f1:.3f}")
print("best val macro F1:", round(best, 3))

epoch  1  train loss 1.969  val macro F1 0.500
epoch  2  train loss 1.090  val macro F1 0.608
epoch  3  train loss 0.752  val macro F1 0.612
epoch  4  train loss 0.525  val macro F1 0.598
epoch  5  train loss 0.350  val macro F1 0.606
epoch  6  train loss 0.270  val macro F1 0.593
epoch  7  train loss 0.186  val macro F1 0.613
epoch  8  train loss 0.138  val macro F1 0.603
epoch  9  train loss 0.119  val macro F1 0.604
epoch 10  train loss 0.090  val macro F1 0.622
epoch 11  train loss 0.074  val macro F1 0.607
epoch 12  train loss 0.075  val macro F1 0.613
best val macro F1: 0.622


In [14]:
# where does the model go wrong? validation results by photo type and by class
import pandas as pd
from pathlib import Path
man = pd.read_csv(f"{DRIVE}/furniture_v2_final_manifest.csv")
room = dict(zip(man["path"].map(lambda p: Path(p).name), man["room_crop"]))
model.load_state_dict(torch.load(MODEL_PATH))
yv, pv = evaluate(val_dl)
is_room = np.array([room[Path(p).name] for p, _ in val_ds.samples])
for name, m in [("room crops", is_room), ("product photos", ~is_room)]:
    print(f"{name:15} n={m.sum():4}  accuracy {np.mean(yv[m] == pv[m]):.2f}")
per = f1_score(yv, pv, average=None, labels=range(len(CLASSES)))
print(pd.Series(per, index=CLASSES).sort_values().round(2).head(8))

room crops      n= 383  accuracy 0.55
product photos  n= 154  accuracy 0.86
storage_cabinet    0.33
table              0.33
nightstand         0.38
coffee_table       0.42
desk               0.44
dresser            0.46
chair              0.50
side_table         0.52
dtype: float64
